# 🔍 Exploratory Data Analysis — Case Study

## Employee Attrition Dataset

---

**Exploratory Data Analysis (EDA)** is arguably the *most important* step in any data-science project. Before building models, tuning hyper-parameters, or deploying pipelines, you must **understand your data**.

### Why EDA matters

| Benefit | Description |
|---|---|
| Spot data-quality issues | Missing values, duplicates, outliers, wrong types |
| Understand distributions | Know what "normal" looks like for each feature |
| Discover relationships | Which variables correlate with the target? |
| Guide feature engineering | Create new features informed by domain knowledge |
| Prevent modelling mistakes | Garbage in → garbage out |

### The EDA mindset

> *"Torture the data, and it will confess to anything."* — Ronald Coase

The goal is **not** to prove a hypothesis — it is to let the data *tell its story*. Stay curious, stay skeptical, and always visualise before you summarise.

## Roadmap

1. **Dataset creation** — build a realistic, deliberately messy dataset
2. **Initial exploration** — first look at shape, types, and summaries
3. **Data-quality assessment** — missing values, duplicates, type issues
4. **Data cleaning** — imputation, deduplication, category fixes, outlier handling
5. **Univariate analysis** — one variable at a time
6. **Bivariate analysis** — feature ↔ target relationships
7. **Correlation analysis** — numeric feature heatmap
8. **Feature engineering ideas** — derive new columns
9. **Key findings & next steps** — wrap-up

---
## 1 · Dataset Creation

We will **synthetically generate** a 1 000-row employee-attrition dataset that mimics the messiness of real-world HR data.

Deliberate problems we will inject:
- ~5–10 % missing values in several columns
- Inconsistent category labels (`"IT"` vs `"I.T."`, `"Bachelors"` vs `"Bachelor's"`)
- Outliers in `salary` and `hours_per_week`
- Duplicate rows

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
pd.set_option("display.max_columns", 20)

print("Libraries loaded ✓")

In [ ]:
np.random.seed(42)
n = 1000

# --- numeric features ---
age = np.random.randint(20, 62, size=n)
salary = np.round(np.random.lognormal(mean=10.8, sigma=0.45, size=n), -2)
years_at_company = np.clip(np.random.exponential(scale=5, size=n), 0, 35).astype(int)
satisfaction_score = np.round(np.clip(np.random.normal(6.5, 2.0, size=n), 1, 10), 1)
hours_per_week = np.round(np.random.normal(42, 6, size=n), 1)

# --- inject outliers ---
outlier_idx_salary = np.random.choice(n, 15, replace=False)
salary[outlier_idx_salary] = np.random.uniform(250_000, 500_000, size=15).round(-2)

outlier_idx_hours = np.random.choice(n, 10, replace=False)
hours_per_week[outlier_idx_hours] = np.random.uniform(65, 85, size=10).round(1)

# --- categorical features (with deliberate inconsistencies) ---
department_clean = ["Engineering", "Sales", "HR", "Marketing", "IT", "Finance"]
department_messy = ["Engineering", "Sales", "HR", "Marketing", "IT", "I.T.",
                    "Finance", "finance", "Engg"]
department = np.random.choice(department_messy, size=n, p=[
    0.22, 0.15, 0.08, 0.12, 0.13, 0.05, 0.12, 0.03, 0.10])

gender = np.random.choice(["Male", "Female", "Other"], size=n, p=[0.52, 0.44, 0.04])

education_messy = ["High School", "Bachelor's", "Bachelors", "Master's",
                   "Masters", "PhD"]
education_level = np.random.choice(education_messy, size=n, p=[
    0.18, 0.30, 0.10, 0.20, 0.07, 0.15])

job_role = np.random.choice(
    ["Analyst", "Manager", "Engineer", "Executive", "Associate", "Director"],
    size=n, p=[0.25, 0.15, 0.25, 0.05, 0.20, 0.10])

# --- target variable (attrition) ---
attrition_prob = 0.18
attrition = np.random.binomial(1, attrition_prob, size=n)

# --- assemble DataFrame ---
df = pd.DataFrame({
    "age": age,
    "salary": salary,
    "years_at_company": years_at_company,
    "satisfaction_score": satisfaction_score,
    "hours_per_week": hours_per_week,
    "department": department,
    "gender": gender,
    "education_level": education_level,
    "job_role": job_role,
    "attrition": attrition,
})

# --- inject missing values (~5-10 %) ---
for col, frac in [("salary", 0.06), ("satisfaction_score", 0.08),
                   ("hours_per_week", 0.05), ("education_level", 0.07),
                   ("department", 0.04)]:
    mask = np.random.choice(n, int(n * frac), replace=False)
    df.loc[mask, col] = np.nan

# --- inject duplicate rows ---
dup_idx = np.random.choice(n, 25, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

print(f"Dataset created: {df.shape[0]} rows × {df.shape[1]} columns (includes duplicates)")

---
## 2 · Initial Exploration

The very first thing we do with *any* new dataset — look at its shape, types, and a few sample rows.

In [ ]:
print("Shape:", df.shape)
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

In [ ]:
print("Unique values per column:")
print(df.nunique())

In [ ]:
print("Data types:")
print(df.dtypes)

### Observations so far

- We have **1 025 rows** (the extra 25 come from the duplicates we injected).
- Several columns have fewer non-null counts → missing values present.
- `department` and `education_level` have more unique values than expected (inconsistent labels).
- `salary` max looks suspiciously high — potential outliers.

---
## 3 · Data-Quality Assessment

Let's quantify the messiness.

### 3.1 Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary = missing_summary[missing_summary.missing_count > 0].sort_values(
    "missing_pct", ascending=False)
print(missing_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cbar=False, cmap="YlOrRd", yticklabels=True, ax=ax)
ax.set_title("Missing-Value Heatmap (yellow = present, red = missing)")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

### 3.2 Duplicate rows

In [ ]:
n_dups = df.duplicated().sum()
print(f"Duplicate rows: {n_dups}")
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)

### 3.3 Inconsistent categories

In [ ]:
print("Department unique values:")
print(df["department"].value_counts(dropna=False))
print()
print("Education level unique values:")
print(df["education_level"].value_counts(dropna=False))

### Quality summary

| Issue | Detail |
|---|---|
| Missing values | `satisfaction_score` (~8 %), `education_level` (~7 %), `salary` (~6 %), `hours_per_week` (~5 %), `department` (~4 %) |
| Duplicates | ~25 rows |
| Inconsistent categories | `department`: "IT" / "I.T." / "Engg" / "finance"; `education_level`: "Bachelor's" / "Bachelors", "Master's" / "Masters" |
| Potential outliers | Very high salaries, very high weekly hours |

---
## 4 · Data Cleaning

We tackle each issue methodically.

### 4.1 Remove duplicates

In [ ]:
print(f"Before: {len(df)} rows")
df = df.drop_duplicates().reset_index(drop=True)
print(f"After:  {len(df)} rows")

### 4.2 Fix inconsistent categories

In [ ]:
dept_map = {
    "I.T.": "IT",
    "finance": "Finance",
    "Engg": "Engineering",
}
df["department"] = df["department"].replace(dept_map)

edu_map = {
    "Bachelors": "Bachelor's",
    "Masters": "Master's",
}
df["education_level"] = df["education_level"].replace(edu_map)

print("Department values after cleaning:")
print(df["department"].value_counts(dropna=False))
print()
print("Education level values after cleaning:")
print(df["education_level"].value_counts(dropna=False))

### 4.3 Handle missing values

Different strategies for different columns:

| Column | Strategy | Rationale |
|---|---|---|
| `salary` | Median imputation | Right-skewed; mean would be pulled by outliers |
| `satisfaction_score` | Median imputation | Roughly symmetric, median is robust |
| `hours_per_week` | Mean imputation | Approximately normal |
| `education_level` | Mode imputation | Categorical — use most frequent value |
| `department` | Mode imputation | Categorical |

In [ ]:
df["salary"] = df["salary"].fillna(df["salary"].median())
df["satisfaction_score"] = df["satisfaction_score"].fillna(df["satisfaction_score"].median())
df["hours_per_week"] = df["hours_per_week"].fillna(df["hours_per_week"].mean().round(1))
df["education_level"] = df["education_level"].fillna(df["education_level"].mode()[0])
df["department"] = df["department"].fillna(df["department"].mode()[0])

print("Remaining missing values:")
print(df.isnull().sum())

### 4.4 Outlier detection & handling (IQR method)

The **Interquartile Range (IQR)** method flags values below Q1 − 1.5·IQR or above Q3 + 1.5·IQR.

In [ ]:
def flag_outliers_iqr(series, name):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ((series < lower) | (series > upper))
    print(f"{name}: Q1={q1:.1f}, Q3={q3:.1f}, IQR={iqr:.1f}, "
          f"bounds=[{lower:.1f}, {upper:.1f}], outliers={outliers.sum()}")
    return lower, upper, outliers

for col in ["salary", "hours_per_week", "age"]:
    lower, upper, mask = flag_outliers_iqr(df[col], col)

In [ ]:
# Cap (winsorize) salary and hours outliers rather than removing rows
for col in ["salary", "hours_per_week"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    after = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: capped {before} outliers (remaining: {after})")

print(f"\nCleaned dataset shape: {df.shape}")

✅ **Cleaning complete.** The dataset is now free of duplicates, inconsistencies, missing values, and extreme outliers. Let's explore it.

---
## 5 · Univariate Analysis

Examine each variable in isolation to understand its distribution.

### 5.1 Numeric features — histograms + box plots

In [ ]:
numeric_cols = ["age", "salary", "years_at_company", "satisfaction_score", "hours_per_week"]

fig, axes = plt.subplots(len(numeric_cols), 2, figsize=(14, 3.2 * len(numeric_cols)))

for i, col in enumerate(numeric_cols):
    # histogram
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color="steelblue", bins=30)
    axes[i, 0].set_title(f"Distribution of {col}")
    axes[i, 0].axvline(df[col].mean(), color="red", ls="--", label=f"mean={df[col].mean():.1f}")
    axes[i, 0].axvline(df[col].median(), color="green", ls="-.", label=f"median={df[col].median():.1f}")
    axes[i, 0].legend(fontsize=8)
    # box plot
    sns.boxplot(x=df[col], ax=axes[i, 1], color="steelblue")
    axes[i, 1].set_title(f"Box plot of {col}")

plt.tight_layout()
plt.show()

### 5.2 Categorical features — bar plots

In [ ]:
cat_cols = ["department", "gender", "education_level", "job_role"]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(y=col, data=df, order=order, ax=axes[i], palette="viridis")
    axes[i].set_title(f"Distribution of {col}")
    for p in axes[i].patches:
        axes[i].annotate(f"{int(p.get_width())}", (p.get_width() + 3, p.get_y() + p.get_height() / 2),
                         va="center", fontsize=9)

plt.tight_layout()
plt.show()

### 5.3 Target variable — attrition

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = df["attrition"].value_counts()
labels = ["Stayed (0)", "Left (1)"]
colors = ["#4CAF50", "#F44336"]
ax.bar(labels, counts.values, color=colors, edgecolor="black")
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha="center", fontweight="bold")
ax.set_title("Attrition Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

print(f"Attrition rate: {df['attrition'].mean():.1%}")

### Univariate takeaways

- **Age** is roughly uniform between 20–61.
- **Salary** is right-skewed (log-normal) even after capping.
- **Years at company** is heavily right-skewed — most employees are relatively new.
- **Satisfaction score** is roughly bell-shaped around 6–7.
- **Hours per week** centres around 42 h.
- **Department** is led by Engineering; **education** by Bachelor's.
- The dataset is **imbalanced**: only ~18 % attrition.

---
## 6 · Bivariate Analysis

How do features relate to our target (`attrition`)?

### 6.1 Numeric features vs. attrition — violin plots

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(18, 5))

df["attrition_label"] = df["attrition"].map({0: "Stayed", 1: "Left"})

for i, col in enumerate(numeric_cols):
    sns.violinplot(x="attrition_label", y=col, data=df, ax=axes[i],
                   order=["Stayed", "Left"],
                   palette={"Stayed": "#4CAF50", "Left": "#F44336"}, inner="quartile")
    axes[i].set_title(col)
    axes[i].set_xlabel("")

df.drop(columns=["attrition_label"], inplace=True)

plt.suptitle("Numeric Features by Attrition", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### 6.2 Categorical features vs. attrition — grouped bar charts

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df["attrition"], normalize="index") * 100
    ct.columns = ["Stayed %", "Left %"]
    ct.sort_values("Left %", ascending=True).plot.barh(
        stacked=True, ax=axes[i], color=["#4CAF50", "#F44336"], edgecolor="black")
    axes[i].set_title(f"Attrition Rate by {col}")
    axes[i].set_xlabel("Percentage")
    axes[i].legend(loc="lower right")

plt.tight_layout()
plt.show()

### 6.3 Salary vs. satisfaction — scatter plot coloured by attrition

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(df["salary"], df["satisfaction_score"],
                     c=df["attrition"], cmap="RdYlGn_r", alpha=0.5, edgecolors="grey", s=30)
ax.set_xlabel("Salary")
ax.set_ylabel("Satisfaction Score")
ax.set_title("Salary vs. Satisfaction (coloured by Attrition)")
cbar = plt.colorbar(scatter, ax=ax, ticks=[0, 1])
cbar.ax.set_yticklabels(["Stayed", "Left"])
plt.tight_layout()
plt.show()

### 6.4 Hours per week vs. years at company

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(x="years_at_company", y="hours_per_week", hue="attrition",
                data=df, palette={0: "#4CAF50", 1: "#F44336"}, alpha=0.6, ax=ax)
ax.set_title("Hours per Week vs. Years at Company")
ax.legend(title="Attrition", labels=["Stayed", "Left"])
plt.tight_layout()
plt.show()

### Bivariate takeaways

- Employees who leave tend to have **lower satisfaction scores**.
- **Salary** differences between stayers and leavers are subtle in this synthetic data.
- Some departments/roles may show higher attrition — worth investigating with larger real-world data.
- The scatter plots don't reveal dramatic clusters — typical for noisy HR data.

---
## 7 · Correlation Analysis

Pearson correlation between all numeric features and the target.

In [ ]:
corr_cols = numeric_cols + ["attrition"]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
target_corr = corr_matrix["attrition"].drop("attrition").sort_values(ascending=False)
print("Feature correlations with attrition:")
print(target_corr.to_string())

### Correlation takeaways

- Because the data is *synthetic and random*, correlations are weak — this is expected. In real data you would see stronger signals.
- Even weak correlations can be useful when combined in a model.
- Correlation ≠ causation — always remember this.

---
## 8 · Feature Engineering Ideas

Create new features that might help a downstream model.

In [ ]:
# Tenure group
bins = [0, 2, 5, 10, 20, 40]
labels_tenure = ["0-2 yr", "3-5 yr", "6-10 yr", "11-20 yr", "20+ yr"]
df["tenure_group"] = pd.cut(df["years_at_company"], bins=bins, labels=labels_tenure, right=True)

# Salary band (use inf so bins always increase regardless of capping)
salary_bins = [0, 40_000, 70_000, 100_000, 150_000, float("inf")]
salary_labels = ["<40k", "40-70k", "70-100k", "100-150k", "150k+"]
df["salary_band"] = pd.cut(df["salary"], bins=salary_bins, labels=salary_labels, right=False)

# Overtime flag (>45 hours per week)
df["overtime_flag"] = (df["hours_per_week"] > 45).astype(int)

print("New features created ✓")
print(df[["years_at_company", "tenure_group", "salary", "salary_band",
          "hours_per_week", "overtime_flag"]].head(10))

### Attrition rate by engineered features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(["tenure_group", "salary_band", "overtime_flag"]):
    ct = pd.crosstab(df[col], df["attrition"], normalize="index") * 100
    ct.columns = ["Stayed %", "Left %"]
    ct["Left %"].plot.bar(ax=axes[i], color="#F44336", edgecolor="black")
    axes[i].set_title(f"Attrition Rate by {col}")
    axes[i].set_ylabel("Attrition %")
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha="right")
    axes[i].axhline(df["attrition"].mean() * 100, color="grey", ls="--", label="overall rate")
    axes[i].legend()

plt.tight_layout()
plt.show()

### Feature engineering notes

- **Tenure groups** bin continuous tenure into interpretable categories.
- **Salary bands** make salary easier to segment.
- **Overtime flag** is a simple binary indicator — more interpretable than raw hours.
- In a real project you might also create interaction terms, ratio features, or aggregate stats per department.

---
## 9 · Key Findings & Next Steps

In [ ]:
# Final cleaned dataset overview
print(f"Final dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Attrition rate: {df['attrition'].mean():.1%}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print()
print("Columns:", list(df.columns))

### 📌 Summary of findings

1. **Data quality**: The raw dataset had ~5–10 % missing values, 25 duplicate rows, inconsistent category labels, and outliers. All issues were resolved.
2. **Distributions**: Salary is right-skewed, tenure is exponentially distributed, satisfaction is roughly normal, and the target is imbalanced (~18 % attrition).
3. **Relationships**: Satisfaction score shows the strongest (though still weak in synthetic data) relationship with attrition. Salary and hours differences between stayers and leavers are subtle.
4. **Feature engineering**: We created `tenure_group`, `salary_band`, and `overtime_flag` — each adds interpretability and may improve model performance.

### 🚀 Recommended next steps

| Step | Description |
|---|---|
| **Encode categoricals** | One-hot or target-encode `department`, `education_level`, etc. |
| **Handle class imbalance** | SMOTE, class weights, or stratified sampling |
| **Baseline model** | Logistic Regression or Random Forest for initial benchmarks |
| **Feature selection** | Use mutual information or recursive feature elimination |
| **Cross-validation** | Stratified k-fold to get robust performance estimates |
| **Interpretability** | SHAP values to explain predictions |

---

*This concludes the EDA case study. The cleaned, feature-engineered DataFrame is ready for modelling.*